# Model Performance Comparison

This notebook compares the performance of:
1. **Baseline XGBoost** (Tabular features only)
2. **HGT (Graph Neural Network)** (Graph structure + embeddings)
3. **Hybrid Model** (XGBoost + GNN embeddings)

## Metrics:
- **AUC-PR** (Area Under Precision-Recall Curve)
- **AUC-ROC** (Area Under ROC Curve)
- **Temporal Performance** (Sliding window analysis)
- **Fraud Detection Rate** by time period

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

## 1. Load Training Results

**Note:** The training scripts need to be updated to save results. For now, we'll create mock data to demonstrate the visualizations.

In [ ]:
# Check if results exist
results_dir = Path("../artifacts/results")
results_dir.mkdir(exist_ok=True)

# Placeholder: Load results from saved CSV/Parquet files
# baseline_results = pl.read_csv(results_dir / "baseline_results.csv")
# gnn_results = pl.read_csv(results_dir / "gnn_results.csv")
# hybrid_results = pl.read_csv(results_dir / "hybrid_results.csv")

# For now, create mock data
print("Creating mock data for demonstration...")
dates = pd.date_range('2024-01-30', '2024-10-28', freq='7D')

np.random.seed(42)
baseline_auc = 0.4 + 0.2 * np.random.rand(len(dates))
gnn_auc = 0.5 + 0.25 * np.random.rand(len(dates))
hybrid_auc = 0.55 + 0.25 * np.random.rand(len(dates))

results_df = pd.DataFrame({
    'date': dates,
    'baseline_auc_pr': baseline_auc,
    'gnn_auc_pr': gnn_auc,
    'hybrid_auc_pr': hybrid_auc
})

results_df.head()

## 2. Temporal Performance Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(results_df['date'], results_df['baseline_auc_pr'], 
        label='Baseline XGBoost', marker='o', linewidth=2, markersize=4)
ax.plot(results_df['date'], results_df['gnn_auc_pr'], 
        label='HGT (GNN)', marker='s', linewidth=2, markersize=4)
ax.plot(results_df['date'], results_df['hybrid_auc_pr'], 
        label='Hybrid (XGB + GNN)', marker='^', linewidth=2, markersize=4)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('AUC-PR', fontsize=12)
ax.set_title('Model Performance Over Time (Sliding Window)', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Average Performance Metrics

In [ ]:
summary = pd.DataFrame({
    'Model': ['Baseline XGBoost', 'HGT (GNN)', 'Hybrid'],
    'Mean AUC-PR': [
        results_df['baseline_auc_pr'].mean(),
        results_df['gnn_auc_pr'].mean(),
        results_df['hybrid_auc_pr'].mean()
    ],
    'Std AUC-PR': [
        results_df['baseline_auc_pr'].std(),
        results_df['gnn_auc_pr'].std(),
        results_df['hybrid_auc_pr'].std()
    ],
    'Max AUC-PR': [
        results_df['baseline_auc_pr'].max(),
        results_df['gnn_auc_pr'].max(),
        results_df['hybrid_auc_pr'].max()
    ],
    'Min AUC-PR': [
        results_df['baseline_auc_pr'].min(),
        results_df['gnn_auc_pr'].min(),
        results_df['hybrid_auc_pr'].min()
    ]
})

summary.round(4)

## 4. Performance Distribution (Box Plot)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

data_to_plot = [
    results_df['baseline_auc_pr'].values,
    results_df['gnn_auc_pr'].values,
    results_df['hybrid_auc_pr'].values
]

bp = ax.boxplot(data_to_plot, labels=['Baseline XGBoost', 'HGT (GNN)', 'Hybrid'],
                patch_artist=True, showmeans=True)

colors = ['lightblue', 'lightgreen', 'lightcoral']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax.set_ylabel('AUC-PR', fontsize=12)
ax.set_title('Model Performance Distribution', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 5. Relative Performance Gain

In [ ]:
# Calculate improvement over baseline
results_df['gnn_improvement'] = ((results_df['gnn_auc_pr'] - results_df['baseline_auc_pr']) 
                                  / results_df['baseline_auc_pr'] * 100)
results_df['hybrid_improvement'] = ((results_df['hybrid_auc_pr'] - results_df['baseline_auc_pr']) 
                                     / results_df['baseline_auc_pr'] * 100)

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(results_df['date'], results_df['gnn_improvement'], 
        label='GNN vs Baseline', marker='o', linewidth=2)
ax.plot(results_df['date'], results_df['hybrid_improvement'], 
        label='Hybrid vs Baseline', marker='s', linewidth=2)

ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Improvement (%)', fontsize=12)
ax.set_title('Relative Performance Improvement Over Baseline', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"Average GNN improvement: {results_df['gnn_improvement'].mean():.2f}%")
print(f"Average Hybrid improvement: {results_df['hybrid_improvement'].mean():.2f}%")

## 6. Concept Drift Analysis

Check if model performance degrades over time (concept drift)

In [ ]:
# Rolling mean to see trends
window_size = 4  # 4 weeks

results_df['baseline_rolling'] = results_df['baseline_auc_pr'].rolling(window=window_size).mean()
results_df['gnn_rolling'] = results_df['gnn_auc_pr'].rolling(window=window_size).mean()
results_df['hybrid_rolling'] = results_df['hybrid_auc_pr'].rolling(window=window_size).mean()

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(results_df['date'], results_df['baseline_rolling'], 
        label='Baseline (4-week avg)', linewidth=3, alpha=0.7)
ax.plot(results_df['date'], results_df['gnn_rolling'], 
        label='GNN (4-week avg)', linewidth=3, alpha=0.7)
ax.plot(results_df['date'], results_df['hybrid_rolling'], 
        label='Hybrid (4-week avg)', linewidth=3, alpha=0.7)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('AUC-PR (Rolling Average)', fontsize=12)
ax.set_title('Performance Trends (Concept Drift Analysis)', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Statistical Significance Test

Perform paired t-test to check if improvements are statistically significant

In [ ]:
from scipy import stats

# Paired t-test: Hybrid vs Baseline
t_stat, p_value = stats.ttest_rel(results_df['hybrid_auc_pr'], results_df['baseline_auc_pr'])

print("=" * 50)
print("Statistical Significance Test: Hybrid vs Baseline")
print("=" * 50)
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"\nSignificant at α=0.05? {'Yes' if p_value < 0.05 else 'No'}")
print(f"Significant at α=0.01? {'Yes' if p_value < 0.01 else 'No'}")

## 8. Export Summary for Paper/Thesis

In [ ]:
# Create LaTeX-ready table
latex_summary = summary.copy()
latex_summary.columns = ['Model', 'Mean', 'Std Dev', 'Max', 'Min']

print("\nLaTeX Table:")
print(latex_summary.to_latex(index=False, float_format="%.4f"))

# Save to CSV
summary.to_csv(results_dir / "model_comparison_summary.csv", index=False)
print(f"\nSaved summary to {results_dir / 'model_comparison_summary.csv'}")